# Silver Layer — Cleaning & Enrichment

Reads from Bronze Delta tables and produces clean, typed, Amsterdam-scoped tables.

Key decisions made here (see docs/why.md for rationale):
- Amsterdam filter applied here (not Bronze) to preserve raw fidelity
- Airbnb zipcodes validated against post_codes.geojson: kept if they are an
  Amsterdam PC4, otherwise replaced via point-in-polygon on the coordinates;
  listings outside Amsterdam are dropped (same rule as the rentals city filter)
- Price outliers capped at IQR upper fence
- Raw string fields (rent, deposit, area) parsed to numeric types

In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("amsterdam-rental-silver").getOrCreate()

try:
    dbutils.fs.ls("/")  # only exists on Databricks
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    nb_path = ctx.notebookPath().get()
    repo_ws_root = "/Workspace" + "/".join(nb_path.split("/")[:-2])
    sys.path.insert(0, f"{repo_ws_root}/src")

    VOLUME_ROOT = "/Volumes/workspace/default/amsterdam_rental"
    DATA_ROOT = f"{VOLUME_ROOT}/data"
    OUTPUT_ROOT = f"{VOLUME_ROOT}/output"
except NameError:
    sys.path.insert(0, os.path.abspath("../src"))
    _repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
    DATA_ROOT = f"{_repo}/data"
    OUTPUT_ROOT = f"{_repo}/data/output"

from amsterdam_rental.geo import PostcodeIndex
from amsterdam_rental.transforms import flatten_list_field, normalise_zipcode, parse_eur, parse_sqm

BRONZE_PATH = f"{OUTPUT_ROOT}/bronze"
SILVER_PATH = f"{OUTPUT_ROOT}/silver"
POSTCODES_GEO = f"{DATA_ROOT}/geo/post_codes.geojson"

AIRBNB_OCCUPANCY_RATE = 0.66  # Amsterdam average (AirDNA)

## Airbnb — silver

In [ ]:
airbnb_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/airbnb")

# Cast numeric columns
airbnb = (
    airbnb_bronze.withColumn("price", F.col("price").cast(T.DoubleType()))
    .withColumn("latitude", F.col("latitude").cast(T.DoubleType()))
    .withColumn("longitude", F.col("longitude").cast(T.DoubleType()))
    .withColumn("accommodates", F.col("accommodates").cast(T.IntegerType()))
    .withColumn("bedrooms", F.col("bedrooms").cast(T.DoubleType()).cast(T.IntegerType()))
    .withColumn("review_scores_value", F.col("review_scores_value").cast(T.DoubleType()))
)

# Normalise zipcode: strip float suffix, keep 4-digit strings
normalise_zipcode_udf = F.udf(normalise_zipcode, T.StringType())
airbnb = airbnb.withColumn("zipcode", normalise_zipcode_udf(F.col("zipcode")))

In [ ]:
# IQR-based price capping (exact percentiles — deterministic, matches the EDA)
q_row = (
    airbnb.filter(F.col("price").isNotNull())
    .selectExpr("percentile(price, 0.25) AS q1", "percentile(price, 0.75) AS q3")
    .first()
)
q1, q3 = q_row["q1"], q_row["q3"]
upper_fence = q3 + 1.5 * (q3 - q1)
print(f"Price IQR fence: Q1={q1:.0f}  Q3={q3:.0f}  upper={upper_fence:.0f}")

outlier_count = airbnb.filter(F.col("price") > upper_fence).count()
print(f"Rows capped: {outlier_count}")

airbnb = airbnb.withColumn(
    "price", F.when(F.col("price") > upper_fence, upper_fence).otherwise(F.col("price"))
)
# Flag rows that were capped so downstream consumers can filter if needed
airbnb = airbnb.withColumn("price_capped", F.col("price") == upper_fence)

In [ ]:
# Zipcode validation + backfill via point-in-polygon
# The geo data ships in the repo as a zip — stage the geojson into the UC
# volume first (same pattern as the bronze source staging).
import os
import shutil
import zipfile

if not os.path.exists(POSTCODES_GEO):
    staged_geo = f"{VOLUME_ROOT}/source/post_codes.geojson"
    if not os.path.exists(staged_geo):
        os.makedirs(os.path.dirname(staged_geo), exist_ok=True)
        with zipfile.ZipFile(f"{repo_ws_root}/data/geo/post_codes.zip") as zf:
            with zf.open("post_codes.geojson") as src, open(staged_geo, "wb") as dst:
                shutil.copyfileobj(src, dst)
    POSTCODES_GEO = staged_geo

# PostcodeIndex is instantiated once and captured in the UDF closure
# (spark.sparkContext.broadcast is not available on serverless / Spark Connect)
pc_index = PostcodeIndex(POSTCODES_GEO)


def resolve_pc4(zipcode, lon, lat):
    # Keep the stated zipcode if it is an Amsterdam PC4; otherwise derive it from
    # the coordinates; None if the listing is outside Amsterdam. Fixes typo /
    # placeholder zipcodes (e.g. '1212AB' on an Oud-Zuid listing) and stops
    # out-of-city postcodes (Hilversum, Diemen, ...) leaking into gold.
    return pc_index.resolve_pc4(zipcode, lon, lat, municipality="Amsterdam")


resolve_pc4_udf = F.udf(resolve_pc4, T.StringType())

airbnb = airbnb.withColumn(
    "zipcode_resolved",
    resolve_pc4_udf(F.col("zipcode"), F.col("longitude"), F.col("latitude")),
)

total = airbnb.count()
stats = airbnb.select(
    F.sum(F.col("zipcode").isNull().cast("int")).alias("backfilled_candidates"),
    F.sum(
        (F.col("zipcode").isNotNull() & (F.col("zipcode") != F.col("zipcode_resolved"))).cast("int")
    ).alias("corrected"),
    F.sum(F.col("zipcode_resolved").isNull().cast("int")).alias("dropped"),
).first()
print(f"Airbnb rows: {total}")
print(f"  missing zipcode (backfill candidates): {stats['backfilled_candidates']}")
print(f"  stated zipcode replaced from coordinates: {stats['corrected']}")
print(f"  dropped (outside Amsterdam / unresolvable): {stats['dropped']}")

airbnb = (
    airbnb.filter(F.col("zipcode_resolved").isNotNull())
    .withColumn("zipcode", F.col("zipcode_resolved"))
    .drop("zipcode_resolved")
)

In [ ]:
# Select and rename to clean schema
airbnb_silver = airbnb.select(
    F.col("zipcode").alias("pc4"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("room_type"),
    F.col("accommodates"),
    F.col("bedrooms"),
    F.col("price").alias("nightly_price_eur"),
    F.col("price_capped"),
    F.col("review_scores_value"),
    F.col("_ingested_at"),
)

print(f"Airbnb silver rows: {airbnb_silver.count()}")
airbnb_silver.printSchema()
airbnb_silver.show(3)

In [ ]:
(airbnb_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/airbnb"))
print("Airbnb silver written.")

## Rentals — silver

In [ ]:
rentals_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/rentals")

# Filter to Amsterdam — business rule, belongs in Silver not Bronze
rentals_ams = rentals_bronze.filter(F.col("city") == "Amsterdam")
print(f"Amsterdam records: {rentals_ams.count()} (of {rentals_bronze.count()} total)")

In [ ]:
# UDFs for string parsing
parse_eur_udf = F.udf(parse_eur, T.DoubleType())
parse_sqm_udf = F.udf(parse_sqm, T.IntegerType())
flatten_udf = F.udf(flatten_list_field, T.StringType())

# Flatten list-wrapped fields and parse string amounts
rentals = (
    rentals_ams
    # Flatten single-element list fields
    .withColumn("id", flatten_udf(F.col("_id")))
    .withColumn("crawled_at", flatten_udf(F.col("crawledAt")))
    .withColumn("first_seen", flatten_udf(F.col("firstSeenAt")))
    .withColumn("last_seen", flatten_udf(F.col("lastSeenAt")))
    # Parse numeric fields
    .withColumn("rent_eur", parse_eur_udf(F.col("rent")))
    .withColumn("deposit_eur", parse_eur_udf(F.col("deposit")))
    .withColumn("additional_costs_eur", parse_eur_udf(F.col("additionalCostsRaw")))
    .withColumn("area_sqm", parse_sqm_udf(F.col("areaSqm")))
    # Cast coordinates
    .withColumn("latitude", F.col("latitude").cast(T.DoubleType()))
    .withColumn("longitude", F.col("longitude").cast(T.DoubleType()))
    # Normalise PC4
    .withColumn("pc4", F.substring(F.col("postalCode"), 1, 4))
)

# Total monthly cost = rent + additional costs (utilities, service fees)
rentals = rentals.withColumn(
    "total_monthly_eur", F.col("rent_eur") + F.coalesce(F.col("additional_costs_eur"), F.lit(0.0))
)

In [ ]:
# Select clean schema
rentals_silver = rentals.select(
    F.col("id"),
    F.col("pc4"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("propertyType").alias("property_type"),
    F.col("area_sqm"),
    F.col("rent_eur"),
    F.col("additional_costs_eur"),
    F.col("total_monthly_eur"),
    F.col("deposit_eur"),
    F.col("furnish"),
    F.col("energyLabel").alias("energy_label"),
    F.col("first_seen"),
    F.col("last_seen"),
    F.col("_ingested_at"),
)

print(f"Rentals silver rows: {rentals_silver.count()}")
rentals_silver.printSchema()
rentals_silver.show(3)

In [ ]:
(rentals_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}/rentals"))
print("Rentals silver written.")

## Silver validation checks

In [ ]:
airbnb_s = spark.read.format("delta").load(f"{SILVER_PATH}/airbnb")
rentals_s = spark.read.format("delta").load(f"{SILVER_PATH}/rentals")

# No nulls in key join column
airbnb_null_pc4 = airbnb_s.filter(F.col("pc4").isNull()).count()
print(f"Airbnb rows with null pc4 after backfill: {airbnb_null_pc4}")
assert airbnb_null_pc4 == 0, "resolve_pc4 should have dropped unresolvable rows"

# Every Airbnb pc4 must be an Amsterdam postcode
non_ams = [
    r["pc4"]
    for r in airbnb_s.select("pc4").distinct().collect()
    if pc_index.municipality(r["pc4"]) != "Amsterdam"
]
assert not non_ams, f"Non-Amsterdam pc4 in Airbnb silver: {non_ams}"

# Rent should be positive
bad_rent = rentals_s.filter(F.col("rent_eur") <= 0).count()
print(f"Rentals rows with zero/negative rent: {bad_rent}")

# Nightly price should be positive
bad_price = airbnb_s.filter(F.col("nightly_price_eur") <= 0).count()
print(f"Airbnb rows with zero/negative price: {bad_price}")

print("\nSilver validation complete.")